In [0]:
# 1. Connection Link Setup
spark.conf.set(
    "fs.azure.account.key.airlineanalyticsprojsrc.dfs.core.windows.net",
    "storage acc key"
)

# 2. ADF ke liye Widget (Parameter) Setup
dbutils.widgets.text("p_file_name", "flights_incremental_1.csv") # Default fallback value
file_name_from_adf = dbutils.widgets.get("p_file_name")

# Dynamic File Path
dynamic_path = f"abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/landing-zone/incremental/{file_name_from_adf}"
print(f"ADF se aayi file read kar rahe hain: {dynamic_path}")

# 3. Read Incoming Incremental Data
new_incremental_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(dynamic_path)

# 4. Applying 10 Advanced Validation Rules (Silver Logic)
from pyspark.sql.functions import col, length, to_date, when

validated_incremental_df = new_incremental_df \
    .dropDuplicates() \
    .filter(col("month").between(1, 12)) \
    .filter(col("day_of_month").between(1, 31)) \
    .filter(col("distance") > 0) \
    .filter(to_date(col("fl_date"), "yyyy-MM-dd").isNotNull()) \
    .filter(length(col("op_unique_carrier")) == 2) \
    .filter((length(col("origin")) == 3) & (length(col("dest")) == 3)) \
    .filter(col("cancelled").isin(0, 1)) \
    .filter(col("diverted").isin(0, 1)) \
    .withColumn("weather_delay", when(col("weather_delay") < 0, 0).otherwise(col("weather_delay"))) \
    .withColumn("carrier_delay", when(col("carrier_delay") < 0, 0).otherwise(col("carrier_delay")))

# Temporary view banayenge MERGE command chalane ke liye
validated_incremental_df.createOrReplaceTempView("src_incremental_data")
print("Data validation complete! Ready for Merge.")

In [0]:
from delta.tables import DeltaTable

silver_table_path = "abfss://airline-flights-data@airlineanalyticsprojsrc.dfs.core.windows.net/silver/silver_flights"
delta_silver_table = DeltaTable.forPath(spark, silver_table_path)

print("Merging (Upserting) new incremental data into Silver Flights Table...")

# MERGE Command execute karna
delta_silver_table.alias("target").merge(
    source = validated_incremental_df.alias("source"),
    condition = (
        (col("target.fl_date") == col("source.fl_date")) & 
        (col("target.op_unique_carrier") == col("source.op_unique_carrier")) & 
        (col("target.op_carrier_fl_num") == col("source.op_carrier_fl_num")) &
        (col("target.origin") == col("source.origin")) & 
        (col("target.dest") == col("source.dest"))
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Incremental Merge in Silver Layer Completed successfully! 🔄")